In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv('../data/data_with_polysemy.csv')
df = df.rename(columns={
    "DATA_FILE": "subject_id",
    "IA_DWELL_TIME": "reading_time",
    "V.Mean.Sum": "valence",
    "A.Mean.Sum": "arousal",
    "D.Mean.Sum": "dominance"
})

In [3]:
all_features = ["SURP_GPT2", "WORD_LEN", "FREQ_WEB", "valence", "arousal", "dominance", "orth_size", "polysemy_count", "answered_correctly"]
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["reading_time"] + all_features)

l1_data = df[df['Native English'] == 1]
train_df_l1, test_df_l1 = train_test_split(l1_data, test_size=0.2, random_state=42)

l2_data = df[df['Native English'] == 0]
train_df_l2, test_df_l2 = train_test_split(l2_data, test_size=0.2, random_state=42)

# Mixed Effect Model

## BaseLine (Surprisal, Length)

In [7]:
def model_mixed_bl(df):
    model_mixed_bl = smf.mixedlm(
    "reading_time ~ SURP_GPT2 + WORD_LEN",
    df,
    groups=df["subject_id"],  # or the grouping variable you're using
    vc_formula={"sentence": "0 + C(sentence)"}  # if you want sentence as random intercept
    ).fit(reml=False)
    print(model_mixed_bl.summary())
    return model_mixed_bl

### L1

In [8]:
model_mixed_bl_l1 = model_mixed_bl(train_df_l1)

          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: reading_time
No. Observations: 90859   Method:             ML          
No. Groups:       69      Scale:              56554.9851  
Min. group size:  1231    Log-Likelihood:     -632496.4495
Max. group size:  1412    Converged:          Yes         
Mean group size:  1316.8                                  
----------------------------------------------------------
               Coef.   Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept       53.834    2.208 24.382 0.000 49.506 58.161
SURP_GPT2        8.347    0.184 45.333 0.000  7.986  8.708
WORD_LEN        31.548    0.379 83.322 0.000 30.806 32.290
sentence Var 16392.675    1.437                           



### L2

In [9]:
model_mixed_bl_l2 = model_mixed_bl(train_df_l2)

           Mixed Linear Model Regression Results
Model:             MixedLM Dependent Variable: reading_time 
No. Observations:  390796  Method:             ML           
No. Groups:        296     Scale:              218492.6329  
Min. group size:   1203    Log-Likelihood:     -2996233.7001
Max. group size:   1409    Converged:          Yes          
Mean group size:   1320.3                                   
------------------------------------------------------------
               Coef.    Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept        58.423    2.403  24.308 0.000 53.713 63.134
SURP_GPT2        20.314    0.177 115.016 0.000 19.968 20.660
WORD_LEN         84.752    0.363 233.691 0.000 84.041 85.463
sentence Var 124872.688    2.313                            



## All Features

In [15]:
def mixed_model_all(df):
    model_mixed_all = smf.mixedlm(
    "reading_time ~ SURP_GPT2 + WORD_LEN + FREQ_WEB + valence + arousal + dominance + orth_size + polysemy_count + answered_correctly",
    df,
    groups=df["subject_id"],  # or the grouping variable you're using
    vc_formula={"sentence": "0 + C(sentence)"}  # if you want sentence as random intercept
    ).fit(reml=False)
    print(model_mixed_all.summary())
    return model_mixed_all

### L1

In [16]:
mixed_model_all_l1 = mixed_model_all(train_df_l1)

              Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   reading_time
No. Observations:     90859     Method:               ML          
No. Groups:           69        Scale:                56321.3335  
Min. group size:      1231      Log-Likelihood:       -632328.6321
Max. group size:      1412      Converged:            Yes         
Mean group size:      1316.8                                      
------------------------------------------------------------------
                     Coef.   Std.Err.   z    P>|z|  [0.025  0.975]
------------------------------------------------------------------
Intercept            -13.839   12.723 -1.088 0.277 -38.776  11.098
SURP_GPT2              6.697    0.213 31.511 0.000   6.280   7.113
WORD_LEN              44.182    2.299 19.216 0.000  39.676  48.688
FREQ_WEB               5.230    0.357 14.649 0.000   4.530   5.930
valence                5.665    1.490  3.803 0.000   2.745   8.585
arousal   

### L2

In [17]:
mixed_model_all_l2 = mixed_model_all(train_df_l2)

               Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    reading_time 
No. Observations:    390796     Method:                ML           
No. Groups:          296        Scale:                 214356.2488  
Min. group size:     1203       Log-Likelihood:        -2992838.7837
Max. group size:     1409       Converged:             Yes          
Mean group size:     1320.3                                         
--------------------------------------------------------------------
                     Coef.    Std.Err.    z    P>|z|  [0.025  0.975]
--------------------------------------------------------------------
Intercept               3.293   11.787   0.279 0.780 -19.810  26.395
SURP_GPT2              12.691    0.202  62.698 0.000  12.294  13.087
WORD_LEN              124.303    2.173  57.213 0.000 120.045 128.561
FREQ_WEB               25.095    0.339  74.014 0.000  24.430  25.759
valence               -16.516    1.426 -11.582 0.0

## Model Selection (Backward StepWise Regression)

In [22]:
def mixed_model_sm(df, model_mixed_all):
    # Initial full set of features
    features = [
        "FREQ_WEB", "WORD_LEN", "SURP_GPT2", "valence", "arousal",
        "dominance", "orth_size", "polysemy_count", "answered_correctly"
    ]

    # Set initial best model
    best_features = features.copy()
    current_best_aic = model_mixed_all.aic
    improvement = True

    while improvement and len(best_features) > 1:
        aic_scores = []
        models = []

        for feature_to_remove in best_features:
            trial_features = [f for f in best_features if f != feature_to_remove]
            formula = "reading_time ~ " + " + ".join(trial_features)

            try:
                md = smf.mixedlm(formula, df, groups="subject_id",
                                 vc_formula={"sentence": "0 + C(sentence)"})
                model = md.fit(reml=False)
                aic_scores.append((model.aic, feature_to_remove, model))
            except Exception as e:
                # If model fails to converge or fit, skip
                print(f"Model failed without {feature_to_remove}: {e}")

        # Find the model with the lowest AIC
        aic_scores.sort()
        best_aic, feature_removed, best_model = aic_scores[0]

        if best_aic < current_best_aic:
            print(f"Removing {feature_removed} improved AIC: {current_best_aic:.2f} → {best_aic:.2f}")
            current_best_aic = best_aic
            best_features.remove(feature_removed)
            model_mixed_sm = best_model  # Update model
        else:
            print("No further improvement in AIC.")
            improvement = False

    # Final selected model summary
    print("\nFinal selected features:", best_features)
    print(model_mixed_sm.summary())
    return model_mixed_sm

### L1

In [23]:
model_mixed_sm_l1 = mixed_model_sm(train_df_l1, mixed_model_all_l1)

Removing answered_correctly improved AIC: 1264681.26 → 1264679.38
Removing dominance improved AIC: 1264679.38 → 1264678.07
No further improvement in AIC.

Final selected features: ['FREQ_WEB', 'WORD_LEN', 'SURP_GPT2', 'valence', 'arousal', 'orth_size', 'polysemy_count']
            Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  reading_time
No. Observations:   90859    Method:              ML          
No. Groups:         69       Scale:               56322.7583  
Min. group size:    1231     Log-Likelihood:      -632329.0373
Max. group size:    1412     Converged:           Yes         
Mean group size:    1316.8                                    
--------------------------------------------------------------
                 Coef.   Std.Err.   z    P>|z|  [0.025  0.975]
--------------------------------------------------------------
Intercept        -19.518   10.324 -1.891 0.059 -39.752   0.716
FREQ_WEB           5.207    0.356 14.629 0.000   

### L2

In [26]:
model_mixed_sm_l2 = mixed_model_sm(train_df_l2, mixed_model_all_l2)

Removing dominance improved AIC: 5985701.57 → 5985699.98
No further improvement in AIC.

Final selected features: ['FREQ_WEB', 'WORD_LEN', 'SURP_GPT2', 'valence', 'arousal', 'orth_size', 'polysemy_count', 'answered_correctly']
               Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    reading_time 
No. Observations:    390796     Method:                ML           
No. Groups:          296        Scale:                 214357.2650  
Min. group size:     1203       Log-Likelihood:        -2992838.9924
Max. group size:     1409       Converged:             Yes          
Mean group size:     1320.3                                         
--------------------------------------------------------------------
                     Coef.    Std.Err.    z    P>|z|  [0.025  0.975]
--------------------------------------------------------------------
Intercept               0.645   11.052   0.058 0.953 -21.016  22.306
FREQ_WEB               25.079 

## Comparison

In [28]:
def comparison(model_mixed_bl, mixed_model_all, model_mixed_sm, test_df):
    comparison_df = pd.DataFrame({
        "Model": ["Only surprisal+length", "All features", "Model Selection"],
        "AIC": [model_mixed_bl.aic, mixed_model_all.aic, model_mixed_sm.aic],
        "BIC": [model_mixed_bl.bic, mixed_model_all.bic, model_mixed_sm.bic],
        "Log-Likelihood": [model_mixed_bl.llf, mixed_model_all.llf, model_mixed_sm.llf]
    })
    print(comparison_df)
    def evaluate_model(model, test_df):
        y_true = test_df["reading_time"]
        y_pred = model.predict(test_df)

        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        return {"MSE": mse, "MAE": mae, "R2": r2}

    results = {
        "BaseLine": evaluate_model(model_mixed_bl, test_df),
        "All Features": evaluate_model(mixed_model_all, test_df),
        "Model Selection": evaluate_model(model_mixed_sm, test_df)
    }

    results_df = pd.DataFrame(results).T
    print(results_df)


### L1

In [29]:
comparison(model_mixed_bl_l1, mixed_model_all_l1, model_mixed_sm_l1, test_df_l1)

                   Model           AIC           BIC  Log-Likelihood
0  Only surprisal+length  1.265003e+06  1.265050e+06  -632496.449472
1           All features  1.264681e+06  1.264794e+06  -632328.632135
2        Model Selection  1.264678e+06  1.264772e+06  -632329.037346
                          MSE         MAE        R2
BaseLine         81252.234623  181.909453  0.129039
All Features     80992.361217  181.756502  0.131825
Model Selection  80985.392099  181.744389  0.131900


### L2

In [30]:
comparison(model_mixed_bl_l2, mixed_model_all_l2, model_mixed_sm_l2, test_df_l2)

                   Model           AIC           BIC  Log-Likelihood
0  Only surprisal+length  5.992477e+06  5.992532e+06   -2.996234e+06
1           All features  5.985702e+06  5.985832e+06   -2.992839e+06
2        Model Selection  5.985700e+06  5.985820e+06   -2.992839e+06
                           MSE         MAE        R2
BaseLine         338877.108770  378.709161  0.190080
All Features     334135.999402  375.734225  0.201412
Model Selection  334132.427433  375.732343  0.201420
